In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

In [2]:
DATA_DIR = Path("../../data/playlist")
RANDOM_SEED = 0

EMB_PATH    = DATA_DIR / "embeddings_pure_bolt.parquet"
LOOKUP_PATH = DATA_DIR / "track_lookup.parquet"
VOCAB_PATH  = DATA_DIR / "training_vocab.parquet"

OUT_META   = DATA_DIR / "umap_subset_meta.parquet"
OUT_COORDS = DATA_DIR / "umap_subset_coords.parquet"

PLAYLIST_COUNT_FIT = 30   # tracks must appear in >= N playlists
ARTIST_CAP_FIT     = 10    # max tracks per artist in fit set

EMB_COLS = [f"e{i}" for i in range(128)]

In [3]:
def load_data():
    print("Loading embeddings …")
    embs = pd.read_parquet(EMB_PATH)                      # track_rowid + e0..e127

    print("Loading track lookup …")
    lookup = pd.read_parquet(LOOKUP_PATH,
                             columns=["track_rowid", "track_name",
                                      "track_popularity", "artist_rowid", "artist_name"])

    print("Loading training vocab (playlist_count) …")
    vocab = pd.read_parquet(VOCAB_PATH,
                            columns=["track_rowid", "playlist_count"])

    print("Joining …")
    df = lookup.merge(embs,  on="track_rowid", how="inner")
    df = df.merge(vocab, on="track_rowid", how="left")
    # tracks not in training vocab have playlist_count NaN → treat as 0
    df["playlist_count"] = df["playlist_count"].fillna(0).astype(int)

    print(f"  Total tracks after join: {len(df):,}")
    return df

df = load_data()

Loading embeddings …
Loading track lookup …
Loading training vocab (playlist_count) …
Joining …
  Total tracks after join: 5,412,049


In [4]:
fit_set = (
    df[df["playlist_count"] >= PLAYLIST_COUNT_FIT]
    .sort_values("playlist_count", ascending=False)
    .groupby("artist_rowid", group_keys=False)
    .head(ARTIST_CAP_FIT)
    .reset_index(drop=True)
)
print(f"Fit set: {len(fit_set):,} tracks")

Fit set: 1,519,559 tracks


# Other entitites

In [5]:
# --- Load missing lookup columns ---
extra   = pd.read_parquet(LOOKUP_PATH, columns=["track_rowid", "album_rowid", "album_name", "label"])
df      = df.merge(extra, on="track_rowid", how="left")
fit_set = fit_set.merge(extra, on="track_rowid", how="left")

print(f"fit_set NaN in album_name: {fit_set['album_name'].isna().sum()}")

fit_set NaN in album_name: 0


In [6]:
# --- Compute entity mean embeddings ---
artist_embs = (
    df.groupby("artist_rowid")
      .agg(
          artist_name=("artist_name", "first"),
          max_popularity=("track_popularity", "max"),
          **{c: (c, "mean") for c in EMB_COLS},
      )
      .reset_index()
      .query("max_popularity >= 40")
)

album_embs = (
    df.groupby("album_rowid")
      .agg(
          album_name=("album_name", "first"),
          artist_name=("artist_name", "first"),
          max_popularity=("track_popularity", "max"),
          n_tracks=("track_rowid", "count"),
          **{c: (c, "mean") for c in EMB_COLS},
      )
      .reset_index()
      .query("max_popularity >= 40 & n_tracks >= 5")
)

print(f"Artists: {len(artist_embs):,}  Albums: {len(album_embs):,}")

Artists: 108,490  Albums: 88,564


In [ ]:
UMAP_PARAMS = dict(
    n_components=2,
    n_neighbors=20,
    min_dist=0.1,
    metric="cosine",
    verbose=True,
    n_jobs=16,
)

fit_matrix = fit_set[EMB_COLS].to_numpy(dtype=np.float32)
print(f"Fitting UMAP on {len(fit_matrix):,} tracks …")
reducer = umap.UMAP(**UMAP_PARAMS)
fit_coords = reducer.fit_transform(fit_matrix)

Fitting UMAP on 1,519,559 tracks …
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=16, n_neighbors=20, verbose=True)
Wed Mar  4 23:24:08 2026 Construct fuzzy simplicial set
Wed Mar  4 23:24:08 2026 Finding Nearest Neighbors
Wed Mar  4 23:24:08 2026 Building RP forest with 64 trees
Wed Mar  4 23:24:30 2026 NN descent for 21 iterations
	 1  /  21
	 2  /  21
	Stopping threshold met -- exiting after 2 iterations
Wed Mar  4 23:25:10 2026 Finished Nearest Neighbor Search
Wed Mar  4 23:25:26 2026 Construct embedding


In [ ]:
print("Transforming artist embeddings …")
artist_coords = reducer.transform(artist_embs[EMB_COLS].to_numpy(dtype=np.float32))

print("Transforming album embeddings …")
album_coords  = reducer.transform(album_embs[EMB_COLS].to_numpy(dtype=np.float32))

def _coords_df(src, type_name, id_col, name_col, coords):
    return pd.DataFrame({
        "entity_type": type_name,
        "entity_id":   src[id_col].astype(str).values,
        "entity_name": src[name_col].values,
        "umap_x":      coords[:, 0].astype(np.float32),
        "umap_y":      coords[:, 1].astype(np.float32),
    })

tracks_c  = _coords_df(fit_set,     "track",  "track_rowid",  "track_name",  fit_coords)
artists_c = _coords_df(artist_embs, "artist", "artist_rowid", "artist_name", artist_coords)
albums_c  = _coords_df(album_embs,  "album",  "album_rowid",  "album_name",  album_coords)

combined_coords = pd.concat([tracks_c, artists_c, albums_c], ignore_index=True)
print(combined_coords.groupby("entity_type").size())

out_path = DATA_DIR / "umap_combined_coords.parquet"
combined_coords.to_parquet(out_path)
print(f"Saved → {out_path}")

In [ ]:
# --- OndaRock notable entities via fuzzy matching ---
from rapidfuzz import process, fuzz

OR_PATH = Path("../../data/ondarock_milestones.csv")
or_df   = pd.read_csv(OR_PATH)

# Artists: store artist_rowid (as str, matching entity_id in combined_coords)
artist_names = artist_embs["artist_name"].str.lower()
NOTABLE_ARTISTS = set()
for or_artist in or_df["artist"].str.lower().unique():
    result = process.extractOne(
        or_artist, artist_names,
        scorer=fuzz.token_sort_ratio, score_cutoff=85,
    )
    if result:
        _, _, idx = result
        rowid = artist_embs.loc[idx, "artist_rowid"]
        NOTABLE_ARTISTS.add(str(rowid))
print(f"Matched {len(NOTABLE_ARTISTS)} notable artists")

# Albums: match on combined "artist - album" key, store album_rowid (as str)
album_embs["_match_key"] = (
    album_embs["artist_name"].str.lower() + " - " +
    album_embs["album_name"].str.lower()
)
album_keys = album_embs["_match_key"]
NOTABLE_ALBUMS = set()
for _, row in or_df.iterrows():
    query  = row["artist"].lower() + " - " + row["album"].lower()
    result = process.extractOne(
        query, album_keys,
        scorer=fuzz.token_sort_ratio, score_cutoff=80,
    )
    if result:
        _, _, idx = result
        rowid = album_embs.loc[idx, "album_rowid"]
        NOTABLE_ALBUMS.add(str(rowid))
print(f"Matched {len(NOTABLE_ALBUMS)} notable albums")

In [ ]:
# --- Multi-entity UMAP: Cell 5 — Visualize tracks + artists + albums + labels ---
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.style.use("dark_background")

tc  = combined_coords[combined_coords.entity_type == "track"]
ac  = combined_coords[combined_coords.entity_type == "artist"]
alc = combined_coords[combined_coords.entity_type == "album"]

fig, ax = plt.subplots(figsize=(100, 100))

# Layer 1 — track density
_, _, _, img = ax.hist2d(tc.umap_x, tc.umap_y, bins=2000, cmap="inferno",
                         norm=mcolors.PowerNorm(gamma=0.3))
plt.colorbar(img, ax=ax, shrink=0.2, label="track density")

# Layer 2 — notable albums (deepskyblue dot + small label)
sel_al = alc[alc.entity_id.isin(NOTABLE_ALBUMS)]
ax.scatter(sel_al.umap_x, sel_al.umap_y, s=10, color="deepskyblue", zorder=6, linewidths=0)
for _, row in sel_al.iterrows():
    ax.annotate(
        row.entity_name,
        (row.umap_x, row.umap_y),
        fontsize=2, color="deepskyblue",
        xytext=(4, 4), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.4, linewidth=0),
    )

# Layer 3 — labeled artists (white dot + bold label)
sel_a = ac[ac.entity_id.isin(NOTABLE_ARTISTS)]
ax.scatter(sel_a.umap_x, sel_a.umap_y, s=5, color="white", zorder=5, linewidths=0)
for _, row in sel_a.iterrows():
    ax.annotate(
        row.entity_name,
        (row.umap_x, row.umap_y),
        fontsize=4, fontweight="bold", color="white",
        xytext=(4, 4), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5, linewidth=0),
    )

ax.set_axis_off()
fig.tight_layout()
fig.savefig("umap_combined_plot.png", dpi=300, bbox_inches="tight")
plt.show()